# 00 — First look at RAVDESS (one actor)

Goal: before writing the feature extractor, check our assumptions on **real** data:
archive layout, number of AV clips, video/audio format, and whether the Haar
face detector finds faces. Nothing here is saved; it only prints facts.

Kaggle settings (right panel): **Accelerator = GPU T4**, **Internet = On**.

## 1. Environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working /tmp | cat
!which ffmpeg ffprobe

## 2. Get the code and run the unit tests

In [ ]:
# Replace with your GitHub username once the repo is pushed.
REPO_URL = "https://github.com/YOUR_USERNAME/ravdess-av-baseline.git"

import os
if not os.path.exists("ravdess-av-baseline"):
    !git clone -q {REPO_URL}
%cd ravdess-av-baseline
!python -m pytest -q

## 3. Download one actor archive (~550 MB)

Archives go to `/tmp` (scratch space, not saved), never to `/kaggle/working`.
`wget -c` resumes a partially downloaded file.

In [ ]:
ACTOR = 1
ZIP_NAME = f"Video_Speech_Actor_{ACTOR:02d}.zip"
URL = f"https://zenodo.org/records/1188976/files/{ZIP_NAME}?download=1"
RAW_DIR = "/tmp/ravdess_raw"
ZIP_PATH = f"{RAW_DIR}/{ZIP_NAME}"

os.makedirs(RAW_DIR, exist_ok=True)
!wget -c -q --show-progress -O {ZIP_PATH} "{URL}"
!ls -lh {ZIP_PATH}

## 4. What is inside the archive?

In [ ]:
import zipfile
from collections import Counter
from ravdess_av.data import av_speech_members, parse_filename

with zipfile.ZipFile(ZIP_PATH) as zf:
    names = zf.namelist()

print("total entries:", len(names))
print("first entries:", names[:5])

mp4 = [n for n in names if n.lower().endswith(".mp4") and "__MACOSX" not in n]
print("modality counts:", Counter(n.split("/")[-1][:2] for n in mp4))

members = av_speech_members(names)
print("AV speech clips:", len(members))                     # expect 60
print("per emotion:", Counter(c.emotion_name for _, c in members))

## 5. Look at one clip: format of video and audio

In [ ]:
member, info = members[0]
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extract(member, RAW_DIR)
CLIP = f"{RAW_DIR}/{member}"
print(info)

!ffprobe -v error -show_entries stream=codec_type,codec_name,width,height,r_frame_rate,sample_rate,channels:format=duration -of compact "{CLIP}"

## 6. Sample 8 frames and run the Haar face detector

Green box = detected face (with ~20% margin). Red title = no face found,
which would trigger the centre-crop fallback in the extractor.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

cap = cv2.VideoCapture(CLIP)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
idx = np.linspace(0, n_frames - 1, 8).round().astype(int)
cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

fig, axes = plt.subplots(2, 4, figsize=(16, 5))
for ax, i in zip(axes.flat, idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
    ok, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    if len(faces):
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])   # largest face
        m = int(0.2 * w)
        cv2.rectangle(rgb, (x - m, y - m), (x + w + m, y + h + m), (0, 255, 0), 4)
    ax.imshow(rgb); ax.axis("off")
    ax.set_title(f"frame {i}: {len(faces)} face(s)", color="black" if len(faces) else "red")
cap.release()
print("total frames:", n_frames)
plt.tight_layout(); plt.show()

## 7. Clean up

In [ ]:
!rm -rf {RAW_DIR}
!df -h /tmp | cat